In [1]:
import csv
from pathlib import Path
from collections import Counter

orig_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408.csv')
copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')

with orig_path.open('r', encoding='utf-8-sig', newline='') as f:
    orig_rows = list(csv.DictReader(f))

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    copy_rows = list(csv.DictReader(f))

orig_compare_cols = [
    'Comparación Forma de Pago',
    'Comparación Cuotas',
    'Comparación Monto',
    'Comparación Total',
]
orig_data_cols = [c for c in orig_rows[0].keys() if c not in orig_compare_cols]
copy_cols = list(copy_rows[0].keys())

print('Original rows:', len(orig_rows))
print('Copy rows:', len(copy_rows))
print('Original data cols:', len(orig_data_cols))
print('Copy cols:', len(copy_cols))
print('Column sets equal:', orig_data_cols == copy_cols)

orig_by_student = {row['ID Estudiante']: {k: row.get(k, '') for k in orig_data_cols} for row in orig_rows}
copy_by_student = {row['ID Estudiante']: row for row in copy_rows}

all_student_ids = sorted(set(orig_by_student) | set(copy_by_student))
changed = []
col_counter = Counter()
for student_id in all_student_ids:
    orig = orig_by_student.get(student_id)
    copy = copy_by_student.get(student_id)
    if orig is None or copy is None:
        changed.append({
            'ID Estudiante': student_id,
            'missing_in': 'original' if orig is None else 'copy',
        })
        continue
    diffs = []
    for col in orig_data_cols:
        old = '' if orig.get(col) is None else str(orig.get(col))
        new = '' if copy.get(col) is None else str(copy.get(col))
        if old != new:
            diffs.append((col, old, new))
            col_counter[col] += 1
    if diffs:
        changed.append({
            'ID Matrícula': copy['ID Matrícula'],
            'ID Estudiante': student_id,
            'Estudiante': copy['Estudiante'],
            'diffs': diffs,
        })

print('Changed rows:', len(changed))
print('Changed columns counts:', dict(col_counter))
print('--- CHANGED ROWS ---')
for row in changed:
    print(f"{row.get('ID Matrícula', '')} | {row['ID Estudiante']} | {row.get('Estudiante', '')}")
    if 'diffs' in row:
        for col, old, new in row['diffs']:
            print(f"  - {col}: {old!r} -> {new!r}")
    else:
        print(f"  - missing_in: {row['missing_in']}")


Original rows: 432
Copy rows: 432
Original data cols: 18
Copy cols: 18
Column sets equal: True
Changed rows: 83
Changed columns counts: {'Aranceles: Forma de Pago': 25, 'Matrícula: N° Cuotas': 60, 'Matrícula: Monto por Cuota': 53, 'Matrícula: Total Anual': 52, 'Aranceles: N° Cuotas': 11, 'Aranceles: Monto Cuota Mínimo': 9, 'Aranceles: Monto Cuota Máximo': 9, 'Aranceles: Total': 9, 'Aranceles: N° Cuota Mínima': 9, 'Aranceles: N° Cuota Máxima': 9, 'Matrícula: Forma de Pago': 2}
--- CHANGED ROWS ---
f1b103fb-4400-471f-8d2a-4dd789326207 | 04e6c1a6-e72c-4ccc-aa13-9e900fa3be97 | ELEONORA PUEBLA MORENO
  - Aranceles: Forma de Pago: 'PAGARE|TRANSFERENCIA' -> 'PAGARE'
8e1acbd6-5724-4d81-a55a-b6639b59afde | 066d269b-8140-4924-8b28-17e2a66df04b | MAYA AURORA MERA GARCÍA
  - Matrícula: N° Cuotas: '' -> '0'
  - Matrícula: Monto por Cuota: '' -> '0'
  - Matrícula: Total Anual: '' -> '0'
fa19b9ee-9765-487f-b052-b96d8ca2828f | 08b2dd2d-0739-46fc-8a4d-19e9013ca0de | FLORENCIA AYALÉN TERUEL
  - Arancele

In [2]:
import json
from pathlib import Path

out_json = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_compare_csv_changes.json')
out_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_compare_csv_changes.csv')

serializable = []
for row in changed:
    serializable.append({
        'ID Matrícula': row.get('ID Matrícula', ''),
        'ID Estudiante': row['ID Estudiante'],
        'Estudiante': row.get('Estudiante', ''),
        'diffs': [
            {'column': col, 'original': old, 'copy': new}
            for col, old, new in row.get('diffs', [])
        ],
    })

out_json.write_text(json.dumps({
    'changed_rows': len(serializable),
    'column_counts': dict(col_counter),
    'rows': serializable,
}, ensure_ascii=False, indent=2), encoding='utf-8')

with out_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID Matrícula', 'ID Estudiante', 'Estudiante', 'column', 'original', 'copy'])
    for row in serializable:
        for diff in row['diffs']:
            writer.writerow([
                row['ID Matrícula'],
                row['ID Estudiante'],
                row['Estudiante'],
                diff['column'],
                diff['original'],
                diff['copy'],
            ])

print(out_json)
print(out_csv)
print('changed_rows =', len(serializable))
print('column_counts =', dict(col_counter))

c:\Meik_Apps\winterhill-cobros-1.1\tmp_compare_csv_changes.json
c:\Meik_Apps\winterhill-cobros-1.1\tmp_compare_csv_changes.csv
changed_rows = 83
column_counts = {'Aranceles: Forma de Pago': 25, 'Matrícula: N° Cuotas': 60, 'Matrícula: Monto por Cuota': 53, 'Matrícula: Total Anual': 52, 'Aranceles: N° Cuotas': 11, 'Aranceles: Monto Cuota Mínimo': 9, 'Aranceles: Monto Cuota Máximo': 9, 'Aranceles: Total': 9, 'Aranceles: N° Cuota Mínima': 9, 'Aranceles: N° Cuota Máxima': 9, 'Matrícula: Forma de Pago': 2}


In [3]:
from collections import defaultdict

copy_by_student = {row['ID Estudiante']: row for row in copy_rows}
actions = {
    'matricula_updates': [],
    'fee_target_updates': [],
}
for row in changed:
    student_id = row['ID Estudiante']
    copy_row = copy_by_student[student_id]
    changed_cols = {col for col, _, _ in row['diffs']}
    if any(col.startswith('Matrícula:') for col in changed_cols):
        actions['matricula_updates'].append({
            'enrollment_id': copy_row['ID Matrícula'],
            'student_id': student_id,
            'student_name': copy_row['Estudiante'],
            'payment_method': copy_row['Matrícula: Forma de Pago'],
            'prioritario': copy_row['Matrícula: Prioritario'],
            'cuotas': copy_row['Matrícula: N° Cuotas'],
            'monto_cuota': copy_row['Matrícula: Monto por Cuota'],
            'total_anual': copy_row['Matrícula: Total Anual'],
            'changed_cols': sorted(col for col in changed_cols if col.startswith('Matrícula:')),
        })
    if any(col.startswith('Aranceles:') for col in changed_cols):
        actions['fee_target_updates'].append({
            'enrollment_id': copy_row['ID Matrícula'],
            'student_id': student_id,
            'student_name': copy_row['Estudiante'],
            'target_fee_count': copy_row['Aranceles: N° Cuotas'],
            'target_payment_method': copy_row['Aranceles: Forma de Pago'],
            'target_min_amount': copy_row['Aranceles: Monto Cuota Mínimo'],
            'target_max_amount': copy_row['Aranceles: Monto Cuota Máximo'],
            'target_total': copy_row['Aranceles: Total'],
            'target_min_installment': copy_row['Aranceles: N° Cuota Mínima'],
            'target_max_installment': copy_row['Aranceles: N° Cuota Máxima'],
            'changed_cols': sorted(col for col in changed_cols if col.startswith('Aranceles:')),
        })

Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_action_plan.json').write_text(
    json.dumps(actions, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('matricula_updates =', len(actions['matricula_updates']))
print('fee_target_updates =', len(actions['fee_target_updates']))
print(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_action_plan.json')

matricula_updates = 66
fee_target_updates = 26
c:\Meik_Apps\winterhill-cobros-1.1\tmp_action_plan.json


In [4]:
matricula_payload = [
    {
        'enrollment_id': item['enrollment_id'],
        'student_id': item['student_id'],
        'payment_method': item['payment_method'],
        'prioritario': item['prioritario'],
        'cuotas': item['cuotas'],
        'monto_cuota': item['monto_cuota'],
        'total_anual': item['total_anual'],
    }
    for item in actions['matricula_updates']
]

fee_payload = [
    {
        'enrollment_id': item['enrollment_id'],
        'student_id': item['student_id'],
        'target_fee_count': item['target_fee_count'],
        'target_payment_method': item['target_payment_method'],
        'target_min_amount': item['target_min_amount'],
        'target_total': item['target_total'],
        'target_min_installment': item['target_min_installment'],
        'target_max_installment': item['target_max_installment'],
    }
    for item in actions['fee_target_updates']
]

matricula_json = json.dumps(matricula_payload, ensure_ascii=False)
fee_json = json.dumps(fee_payload, ensure_ascii=False)

sql = f"""
begin;

with
matricula_src as (
  select *
  from jsonb_to_recordset('{matricula_json}'::jsonb) as s(
    enrollment_id uuid,
    student_id uuid,
    payment_method text,
    prioritario text,
    cuotas text,
    monto_cuota text,
    total_anual text
  )
),
matricula_norm as (
  select
    enrollment_id,
    student_id,
    student_id::text as student_key,
    upper(nullif(trim(payment_method), '')) as payment_method,
    lower(coalesce(prioritario, 'false')) = 'true' as prioritario,
    case when nullif(trim(coalesce(cuotas, '')), '') is null then null else trim(cuotas)::int end as cuotas,
    case when nullif(trim(coalesce(monto_cuota, '')), '') is null then null else trim(monto_cuota)::numeric end as monto_cuota,
    case when nullif(trim(coalesce(total_anual, '')), '') is null then null else trim(total_anual)::numeric end as total_anual
  from matricula_src
),
enrollment_counts as (
  select enrollment_id, count(*) as student_count
  from public.enrollment_students
  group by enrollment_id
),
matricula_prepared as (
  select
    m.*, 
    ec.student_count,
    coalesce(e.meta, '{{}}'::jsonb) as current_meta,
    coalesce(e.meta -> 'per_student_economic' -> m.student_key, '{{}}'::jsonb) as current_student_economic,
    (
      select coalesce(
        jsonb_agg(
          jsonb_build_object(
            'numero', gs,
            'amount', coalesce(m.monto_cuota, 0),
            'due_date', (date '2026-03-05' + make_interval(months => gs - 1))::date
          ) order by gs
        ),
        '[]'::jsonb
      )
      from generate_series(1, greatest(coalesce(m.cuotas, 0), 0)) as gs
    ) as cuotas_plan
  from matricula_norm m
  join public.enrollments e on e.id = m.enrollment_id and e.year = 2026
  left join enrollment_counts ec on ec.enrollment_id = m.enrollment_id
),
matricula_patched as (
  select
    mp.enrollment_id,
    mp.student_id,
    case
      when coalesce(mp.student_count, 0) = 1 then
        jsonb_set(
          jsonb_set(
            jsonb_set(
              jsonb_set(
                jsonb_set(
                  jsonb_set(
                    jsonb_set(
                      jsonb_set(
                        jsonb_set(
                          jsonb_set(
                            jsonb_set(
                              jsonb_set(
                                jsonb_set(
                                  mp.current_meta,
                                  array['per_student_economic', mp.student_key],
                                  mp.current_student_economic || jsonb_build_object(
                                    'payment_method', mp.payment_method,
                                    'prioritario', mp.prioritario,
                                    'cantidad_cuotas', coalesce(mp.cuotas, 0),
                                    'monto_cuota', coalesce(mp.monto_cuota, 0),
                                    'colegiatura_anual', coalesce(mp.total_anual, 0),
                                    'year_academico', 2026,
                                    'dia_vencimiento', 5
                                  ),
                                  true
                                ),
                                array['per_student_plans', mp.student_key],
                                jsonb_build_object(
                                  'cuotas', mp.cuotas_plan,
                                  'n_cuotas', coalesce(mp.cuotas, 0),
                                  'monto_total', coalesce(mp.total_anual, 0),
                                  'payment_method', case when mp.prioritario then null else mp.payment_method end,
                                  'dia_vencimiento', 5,
                                  'monto_por_cuota', coalesce(mp.monto_cuota, 0),
                                  'primer_vencimiento', '2026-03-05'
                                ),
                                true
                              ),
                              array['payment_method'],
                              to_jsonb(mp.payment_method),
                              true
                            ),
                            array['prioritario'],
                            to_jsonb(mp.prioritario),
                            true
                          ),
                          array['cantidad_cuotas'],
                          to_jsonb(coalesce(mp.cuotas, 0)),
                          true
                        ),
                        array['monto_cuota'],
                        to_jsonb(coalesce(mp.monto_cuota, 0)),
                        true
                      ),
                      array['colegiatura_anual'],
                      to_jsonb(coalesce(mp.total_anual, 0)),
                      true
                    ),
                    array['payment_plan'],
                    jsonb_build_object(
                      'cuotas', mp.cuotas_plan,
                      'n_cuotas', coalesce(mp.cuotas, 0),
                      'monto_total', coalesce(mp.total_anual, 0),
                      'payment_method', case when mp.prioritario then null else mp.payment_method end,
                      'dia_vencimiento', 5,
                      'monto_por_cuota', coalesce(mp.monto_cuota, 0),
                      'primer_vencimiento', '2026-03-05'
                    ),
                    true
                  ),
                  array['forma_pago_pagare'],
                  to_jsonb(mp.payment_method = 'PAGARE'),
                  true
                ),
                array['forma_pago_cheques'],
                to_jsonb(mp.payment_method = 'CHEQUE'),
                true
              ),
              array['forma_pago_tarjeta'],
              to_jsonb(mp.payment_method = 'TARJETA'),
              true
            ),
            array['forma_pago_transferencia'],
            to_jsonb(mp.payment_method = 'TRANSFERENCIA'),
            true
          ),
          array['forma_pago_descuento_planilla'],
          to_jsonb(mp.payment_method = 'PLANILLA'),
          true
        )
      else
        jsonb_set(
          mp.current_meta,
          array['per_student_economic', mp.student_key],
          mp.current_student_economic || jsonb_build_object(
            'payment_method', mp.payment_method,
            'prioritario', mp.prioritario,
            'cantidad_cuotas', coalesce(mp.cuotas, 0),
            'monto_cuota', coalesce(mp.monto_cuota, 0),
            'colegiatura_anual', coalesce(mp.total_anual, 0),
            'year_academico', 2026,
            'dia_vencimiento', 5
          ),
          true
        )
    end as new_meta
  from matricula_prepared mp
),
matricula_updates as (
  update public.enrollments e
  set meta = mp.new_meta,
      updated_at = now()
  from matricula_patched mp
  where e.id = mp.enrollment_id
  returning mp.student_id, e.id as enrollment_id
),
fee_src as (
  select *
  from jsonb_to_recordset('{fee_json}'::jsonb) as s(
    enrollment_id uuid,
    student_id uuid,
    target_fee_count text,
    target_payment_method text,
    target_min_amount text,
    target_total text,
    target_min_installment text,
    target_max_installment text
  )
),
fee_norm as (
  select
    enrollment_id,
    student_id,
    case when nullif(trim(coalesce(target_fee_count, '')), '') is null then 0 else trim(target_fee_count)::int end as target_fee_count,
    upper(nullif(trim(target_payment_method), '')) as target_payment_method,
    case when nullif(trim(coalesce(target_min_amount, '')), '') is null then null else trim(target_min_amount)::numeric end as target_amount,
    case when nullif(trim(coalesce(target_total, '')), '') is null then null else trim(target_total)::numeric end as target_total,
    case when nullif(trim(coalesce(target_min_installment, '')), '') is null then 1 else trim(target_min_installment)::int end as min_installment,
    case
      when nullif(trim(coalesce(target_max_installment, '')), '') is null and nullif(trim(coalesce(target_fee_count, '')), '') is null then 0
      when nullif(trim(coalesce(target_max_installment, '')), '') is null then trim(target_fee_count)::int
      else trim(target_max_installment)::int
    end as max_installment
  from fee_src
),
fee_deletes as (
  delete from public.fee f
  using fee_norm fn
  where f.student_id = fn.student_id
    and f.year_academico = 2026
    and (fn.target_fee_count = 0 or fn.target_payment_method = 'SIN_FEE')
  returning f.id, f.student_id
),
fee_anchor as (
  select distinct
    fn.student_id,
    fn.enrollment_id,
    e.guardian_id,
    s.owner_id,
    s.curso as fee_curso,
    fn.target_payment_method,
    fn.target_amount,
    fn.target_total,
    fn.min_installment,
    fn.max_installment,
    fn.target_fee_count
  from fee_norm fn
  join public.enrollments e on e.id = fn.enrollment_id and e.year = 2026
  join public.students s on s.id = fn.student_id
  where fn.target_fee_count > 0 and fn.target_payment_method <> 'SIN_FEE'
),
fee_desired_rows as (
  select
    fa.student_id,
    fa.guardian_id,
    fa.owner_id,
    fa.fee_curso,
    fa.enrollment_id,
    fa.target_payment_method as payment_method,
    coalesce(fa.target_amount, case when fa.target_fee_count > 0 then round(fa.target_total / fa.target_fee_count, 2) else 0 end, 0) as amount,
    gs as numero_cuota,
    (date '2026-03-05' + make_interval(months => gs - 1))::date as due_date
  from fee_anchor fa
  join lateral generate_series(greatest(fa.min_installment, 1), greatest(fa.max_installment, 0)) gs on true
),
fee_upserts as (
  insert into public.fee (
    student_id,
    guardian_id,
    amount,
    due_date,
    status,
    payment_method,
    owner_id,
    year_academico,
    numero_cuota,
    enrollment_id,
    meta,
    year,
    fee_curso
  )
  select
    dr.student_id,
    dr.guardian_id,
    dr.amount,
    dr.due_date,
    'pending',
    dr.payment_method,
    dr.owner_id,
    2026,
    dr.numero_cuota,
    dr.enrollment_id,
    jsonb_build_object(
      'source', 'reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv',
      'sync_reason', 'csv_review_sync_2026_04_10'
    ),
    2026,
    dr.fee_curso
  from fee_desired_rows dr
  on conflict (student_id, year_academico, numero_cuota)
  do update set
    guardian_id = excluded.guardian_id,
    amount = excluded.amount,
    payment_method = excluded.payment_method,
    owner_id = excluded.owner_id,
    enrollment_id = excluded.enrollment_id,
    year = excluded.year,
    year_academico = excluded.year_academico,
    fee_curso = excluded.fee_curso,
    due_date = coalesce(public.fee.due_date, excluded.due_date),
    meta = coalesce(public.fee.meta, '{{}}'::jsonb) || excluded.meta
  returning student_id, numero_cuota
),
fee_extra_cleanup as (
  delete from public.fee f
  using fee_anchor fa
  where f.student_id = fa.student_id
    and f.year_academico = 2026
    and (f.numero_cuota is null or f.numero_cuota < fa.min_installment or f.numero_cuota > fa.max_installment)
    and coalesce(f.status, 'pending') <> 'paid'
  returning f.id, f.student_id
)
select json_build_object(
  'matricula_updates', (select count(*) from matricula_updates),
  'fee_deleted', (select count(*) from fee_deletes),
  'fee_upserted', (select count(*) from fee_upserts),
  'fee_extra_deleted', (select count(*) from fee_extra_cleanup)
) as summary;

commit;
"""

sql_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_apply_csv_copy_db_changes_20260410.sql')
sql_path.write_text(sql, encoding='utf-8')
print(sql_path)
print('matricula payload:', len(matricula_payload))
print('fee payload:', len(fee_payload))

c:\Meik_Apps\winterhill-cobros-1.1\tmp_apply_csv_copy_db_changes_20260410.sql
matricula payload: 66
fee payload: 26


In [5]:
import subprocess
import urllib.request

project_ref = 'yeotpplgerfpxviqazrn'
sql_file = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_apply_csv_copy_db_changes_20260410.sql')
sql_text = sql_file.read_text(encoding='utf-8')

def get_access_token():
    token = subprocess.check_output(
        ['powershell', '-NoProfile', '-File', 'sql/get_token.ps1'],
        cwd=r'c:\Meik_Apps\winterhill-cobros-1.1',
        text=True,
    ).strip()
    if not token:
        raise RuntimeError('No se pudo obtener token de Supabase')
    return token

def execute_supabase_sql(query_text):
    token = get_access_token()
    payload = json.dumps({'query': query_text}).encode('utf-8')
    req = urllib.request.Request(
        f'https://api.supabase.com/v1/projects/{project_ref}/database/query',
        data=payload,
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
        },
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=120) as response:
        return json.loads(response.read().decode('utf-8'))

dry_run_sql = sql_text.replace('\ncommit;\n', '\nrollback;\n')
dry_run_result = execute_supabase_sql(dry_run_sql)
print('dry_run_result =', dry_run_result)
Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_apply_csv_copy_db_changes_20260410_dry_run_result.json').write_text(
    json.dumps(dry_run_result, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

HTTPError: HTTP Error 403: Forbidden

In [6]:
def sql_quote(value):
    if value is None:
        return 'null'
    text = str(value).replace("'", "''")
    return f"'{text}'"

matricula_zero = []
matricula_special = []
for item in actions['matricula_updates']:
    payment_method = item['payment_method']
    cuotas = item['cuotas'] or '0'
    monto = item['monto_cuota'] or '0'
    total = item['total_anual'] or '0'
    if payment_method == 'PRIORITARIO' and cuotas == '0' and monto == '0' and total == '0':
        matricula_zero.append(item)
    else:
        matricula_special.append(item)

fee_zero = []
fee_nonzero = []
matricula_by_student = {item['student_id']: item for item in actions['matricula_updates']}
for item in actions['fee_target_updates']:
    count = int(item['target_fee_count'] or 0)
    method = item['target_payment_method'] or ''
    min_amount = item['target_min_amount'] or None
    total = item['target_total'] or None
    max_installment = int(item['target_max_installment']) if item['target_max_installment'] else 0
    min_installment = int(item['target_min_installment']) if item['target_min_installment'] else 1
    matricula = matricula_by_student.get(item['student_id'])
    if count == 0 or method == 'SIN_FEE':
        fee_zero.append(item)
        continue
    effective_amount = min_amount
    effective_total = total
    if count > 1:
        inconsistent = (
            min_amount is None
            or total is None
            or max_installment < count
        )
        if not inconsistent:
            try:
                inconsistent = float(min_amount) * count != float(total)
            except ValueError:
                inconsistent = True
        if inconsistent and matricula:
            effective_amount = matricula['monto_cuota']
            effective_total = matricula['total_anual']
            min_installment = 1
            max_installment = count
    fee_nonzero.append({
        'enrollment_id': item['enrollment_id'],
        'student_id': item['student_id'],
        'target_fee_count': count,
        'target_payment_method': method,
        'target_min_amount': effective_amount,
        'target_total': effective_total,
        'target_min_installment': min_installment,
        'target_max_installment': max_installment,
    })

zero_ids = ',\n    '.join(sql_quote(item['student_id']) + '::uuid' for item in matricula_zero)
matricula_zero_sql = f"""
with target_students as (
  select unnest(array[
    {zero_ids}
  ]) as student_id
), patched as (
  select
    e.id as enrollment_id,
    s.id as student_id,
    s.id::text as student_key,
    coalesce(e.meta, '{{}}'::jsonb) as current_meta,
    coalesce(e.meta -> 'per_student_economic' -> (s.id::text), '{{}}'::jsonb) as current_student_economic,
    (
      select '[]'::jsonb
    ) as cuotas_plan,
    count(es2.student_id) over (partition by e.id) as student_count
  from target_students ts
  join public.students s on s.id = ts.student_id
  join public.enrollment_students es on es.student_id = s.id
  join public.enrollments e on e.id = es.enrollment_id and e.year = 2026
  join public.enrollment_students es2 on es2.enrollment_id = e.id
)
update public.enrollments e
set meta = case
    when p.student_count = 1 then
      jsonb_set(
        jsonb_set(
          jsonb_set(
            jsonb_set(
              jsonb_set(
                jsonb_set(
                  jsonb_set(
                    jsonb_set(
                      jsonb_set(
                        jsonb_set(
                          jsonb_set(
                            p.current_meta,
                            array['per_student_economic', p.student_key],
                            p.current_student_economic || jsonb_build_object(
                              'payment_method', 'PRIORITARIO',
                              'prioritario', true,
                              'cantidad_cuotas', 0,
                              'monto_cuota', 0,
                              'colegiatura_anual', 0,
                              'year_academico', 2026,
                              'dia_vencimiento', 5
                            ),
                            true
                          ),
                          array['per_student_plans', p.student_key],
                          jsonb_build_object(
                            'cuotas', '[]'::jsonb,
                            'n_cuotas', 0,
                            'monto_total', 0,
                            'payment_method', null,
                            'dia_vencimiento', 5,
                            'monto_por_cuota', 0,
                            'primer_vencimiento', '2026-03-05'
                          ),
                          true
                        ),
                        array['payment_method'], to_jsonb('PRIORITARIO'::text), true
                      ),
                      array['prioritario'], to_jsonb(true), true
                    ),
                    array['cantidad_cuotas'], to_jsonb(0), true
                  ),
                  array['monto_cuota'], to_jsonb(0), true
                ),
                array['colegiatura_anual'], to_jsonb(0), true
              ),
              array['payment_plan'], jsonb_build_object(
                'cuotas', '[]'::jsonb,
                'n_cuotas', 0,
                'monto_total', 0,
                'payment_method', null,
                'dia_vencimiento', 5,
                'monto_por_cuota', 0,
                'primer_vencimiento', '2026-03-05'
              ), true
            ),
            array['forma_pago_pagare'], to_jsonb(false), true
          ),
          array['forma_pago_cheques'], to_jsonb(false), true
        ),
        array['forma_pago_transferencia'], to_jsonb(false), true
      )
    else
      jsonb_set(
        jsonb_set(
          p.current_meta,
          array['per_student_economic', p.student_key],
          p.current_student_economic || jsonb_build_object(
            'payment_method', 'PRIORITARIO',
            'prioritario', true,
            'cantidad_cuotas', 0,
            'monto_cuota', 0,
            'colegiatura_anual', 0,
            'year_academico', 2026,
            'dia_vencimiento', 5
          ),
          true
        ),
        array['per_student_plans', p.student_key],
        jsonb_build_object(
          'cuotas', '[]'::jsonb,
          'n_cuotas', 0,
          'monto_total', 0,
          'payment_method', null,
          'dia_vencimiento', 5,
          'monto_por_cuota', 0,
          'primer_vencimiento', '2026-03-05'
        ),
        true
      )
  end,
  updated_at = now()
from patched p
where e.id = p.enrollment_id;
""".strip()

mat_special_values = ',\n  '.join(
    f"({sql_quote(item['enrollment_id'])}::uuid, {sql_quote(item['student_id'])}::uuid, {sql_quote(item['payment_method'])}, {str(item['payment_method'] == 'PRIORITARIO').lower()}, {item['cuotas'] or '0'}, {item['monto_cuota'] or '0'}, {item['total_anual'] or '0'})"
    for item in matricula_special
)
matricula_special_sql = f"""
with src(enrollment_id, student_id, payment_method, prioritario, cuotas, monto_cuota, total_anual) as (
  values
  {mat_special_values}
), prepared as (
  select
    src.*, src.student_id::text as student_key,
    coalesce(e.meta, '{{}}'::jsonb) as current_meta,
    coalesce(e.meta -> 'per_student_economic' -> (src.student_id::text), '{{}}'::jsonb) as current_student_economic,
    count(es2.student_id) over (partition by e.id) as student_count,
    (
      select coalesce(jsonb_agg(jsonb_build_object('numero', gs, 'amount', src.monto_cuota, 'due_date', (date '2026-03-05' + make_interval(months => gs - 1))::date) order by gs), '[]'::jsonb)
      from generate_series(1, greatest(src.cuotas, 0)) gs
    ) as cuotas_plan
  from src
  join public.enrollments e on e.id = src.enrollment_id and e.year = 2026
  join public.enrollment_students es2 on es2.enrollment_id = e.id
)
update public.enrollments e
set meta = case
    when p.student_count = 1 then
      jsonb_set(
        jsonb_set(
          jsonb_set(
            jsonb_set(
              jsonb_set(
                jsonb_set(
                  jsonb_set(
                    jsonb_set(
                      jsonb_set(
                        jsonb_set(
                          jsonb_set(
                            jsonb_set(
                              p.current_meta,
                              array['per_student_economic', p.student_key],
                              p.current_student_economic || jsonb_build_object(
                                'payment_method', p.payment_method,
                                'prioritario', p.prioritario,
                                'cantidad_cuotas', p.cuotas,
                                'monto_cuota', p.monto_cuota,
                                'colegiatura_anual', p.total_anual,
                                'year_academico', 2026,
                                'dia_vencimiento', 5
                              ),
                              true
                            ),
                            array['per_student_plans', p.student_key],
                            jsonb_build_object(
                              'cuotas', p.cuotas_plan,
                              'n_cuotas', p.cuotas,
                              'monto_total', p.total_anual,
                              'payment_method', case when p.prioritario then null else p.payment_method end,
                              'dia_vencimiento', 5,
                              'monto_por_cuota', p.monto_cuota,
                              'primer_vencimiento', '2026-03-05'
                            ),
                            true
                          ),
                          array['payment_method'], to_jsonb(p.payment_method), true
                        ),
                        array['prioritario'], to_jsonb(p.prioritario), true
                      ),
                      array['cantidad_cuotas'], to_jsonb(p.cuotas), true
                    ),
                    array['monto_cuota'], to_jsonb(p.monto_cuota), true
                  ),
                  array['colegiatura_anual'], to_jsonb(p.total_anual), true
                ),
                array['payment_plan'], jsonb_build_object(
                  'cuotas', p.cuotas_plan,
                  'n_cuotas', p.cuotas,
                  'monto_total', p.total_anual,
                  'payment_method', case when p.prioritario then null else p.payment_method end,
                  'dia_vencimiento', 5,
                  'monto_por_cuota', p.monto_cuota,
                  'primer_vencimiento', '2026-03-05'
                ), true
              ),
              array['forma_pago_pagare'], to_jsonb(p.payment_method = 'PAGARE'), true
            ),
            array['forma_pago_cheques'], to_jsonb(p.payment_method = 'CHEQUE'), true
          ),
          array['forma_pago_transferencia'], to_jsonb(p.payment_method = 'TRANSFERENCIA'), true
        ),
        array['forma_pago_descuento_planilla'], to_jsonb(p.payment_method = 'PLANILLA'), true
      )
    else
      jsonb_set(
        jsonb_set(
          p.current_meta,
          array['per_student_economic', p.student_key],
          p.current_student_economic || jsonb_build_object(
            'payment_method', p.payment_method,
            'prioritario', p.prioritario,
            'cantidad_cuotas', p.cuotas,
            'monto_cuota', p.monto_cuota,
            'colegiatura_anual', p.total_anual,
            'year_academico', 2026,
            'dia_vencimiento', 5
          ), true
        ),
        array['per_student_plans', p.student_key],
        jsonb_build_object(
          'cuotas', p.cuotas_plan,
          'n_cuotas', p.cuotas,
          'monto_total', p.total_anual,
          'payment_method', case when p.prioritario then null else p.payment_method end,
          'dia_vencimiento', 5,
          'monto_por_cuota', p.monto_cuota,
          'primer_vencimiento', '2026-03-05'
        ), true
      )
  end,
  updated_at = now()
from prepared p
where e.id = p.enrollment_id;
""".strip()

fee_zero_ids = ',\n    '.join(sql_quote(item['student_id']) + '::uuid' for item in fee_zero)
fee_delete_sql = f"""
delete from public.fee
where year_academico = 2026
  and student_id = any(array[
    {fee_zero_ids}
  ]);
""".strip()

fee_values = ',\n  '.join(
    f"({sql_quote(item['enrollment_id'])}::uuid, {sql_quote(item['student_id'])}::uuid, {item['target_fee_count']}, {sql_quote(item['target_payment_method'])}, {item['target_min_amount'] or 'null'}, {item['target_total'] or 'null'}, {item['target_min_installment']}, {item['target_max_installment']})"
    for item in fee_nonzero
)
fee_upsert_sql = f"""
with src(enrollment_id, student_id, target_fee_count, payment_method, amount, total_amount, min_installment, max_installment) as (
  values
  {fee_values}
), desired_rows as (
  select
    src.student_id,
    e.guardian_id,
    s.owner_id,
    s.curso as fee_curso,
    src.enrollment_id,
    src.payment_method,
    src.amount,
    gs as numero_cuota,
    (date '2026-03-05' + make_interval(months => gs - 1))::date as due_date
  from src
  join public.enrollments e on e.id = src.enrollment_id and e.year = 2026
  join public.students s on s.id = src.student_id
  join lateral generate_series(src.min_installment, src.max_installment) gs on true
)
insert into public.fee (
  student_id,
  guardian_id,
  amount,
  due_date,
  status,
  payment_method,
  owner_id,
  year_academico,
  numero_cuota,
  enrollment_id,
  meta,
  year,
  fee_curso
)
select
  dr.student_id,
  dr.guardian_id,
  dr.amount,
  dr.due_date,
  'pending',
  dr.payment_method,
  dr.owner_id,
  2026,
  dr.numero_cuota,
  dr.enrollment_id,
  jsonb_build_object(
    'source', 'reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv',
    'sync_reason', 'csv_review_sync_2026_04_10'
  ),
  2026,
  dr.fee_curso
from desired_rows dr
on conflict (student_id, year_academico, numero_cuota)
do update set
  guardian_id = excluded.guardian_id,
  amount = excluded.amount,
  payment_method = excluded.payment_method,
  owner_id = excluded.owner_id,
  enrollment_id = excluded.enrollment_id,
  year = excluded.year,
  year_academico = excluded.year_academico,
  fee_curso = excluded.fee_curso,
  due_date = coalesce(public.fee.due_date, excluded.due_date),
  meta = coalesce(public.fee.meta, '{{}}'::jsonb) || excluded.meta;
""".strip()

Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_matricula_zero.sql').write_text(matricula_zero_sql, encoding='utf-8')
Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_matricula_special.sql').write_text(matricula_special_sql, encoding='utf-8')
Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_fee_delete_zero.sql').write_text(fee_delete_sql, encoding='utf-8')
Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_fee_upsert.sql').write_text(fee_upsert_sql, encoding='utf-8')
print('matricula_zero', len(matricula_zero))
print('matricula_special', len(matricula_special))
print('fee_zero', len(fee_zero))
print('fee_nonzero', len(fee_nonzero))

matricula_zero 53
matricula_special 13
fee_zero 7
fee_nonzero 19


In [10]:
import json
import re
from decimal import Decimal, InvalidOperation

live_result_path = Path(r'c:\Users\eduar\AppData\Roaming\Code\User\workspaceStorage\220c91a4418e7742dc4af0311d3a2b08\GitHub.copilot-chat\chat-session-resources\2a8d9d80-8e33-42e8-8041-6ae5e718bc8c\call_UgT1sWUuSBKgu9V6paqiL43r__vscode-1775828510350\content.json')
validation_json = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_post_update_report_validation.json')
validation_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_post_update_report_validation.csv')

compare_cols = [
    'Comparación Forma de Pago',
    'Comparación Cuotas',
    'Comparación Monto',
    'Comparación Total',
]
numeric_cols = {
    'Matrícula: N° Cuotas',
    'Matrícula: Monto por Cuota',
    'Matrícula: Total Anual',
    'Aranceles: N° Cuotas',
    'Aranceles: Monto Cuota Mínimo',
    'Aranceles: Monto Cuota Máximo',
    'Aranceles: Total',
    'Aranceles: N° Cuota Mínima',
    'Aranceles: N° Cuota Máxima',
}

def extract_embedded_json(text):
    candidate = text.strip()
    match = re.search(r'<untrusted-data[^>]*>\s*(\[.*\]|\{.*\})\s*</untrusted-data[^>]*>', candidate, re.S)
    if match:
        return match.group(1)
    bracket_start = candidate.find('[')
    bracket_end = candidate.rfind(']')
    if bracket_start != -1 and bracket_end != -1 and bracket_end > bracket_start:
        return candidate[bracket_start:bracket_end + 1]
    brace_start = candidate.find('{')
    brace_end = candidate.rfind('}')
    if brace_start != -1 and brace_end != -1 and brace_end > brace_start:
        return candidate[brace_start:brace_end + 1]
    return candidate

def find_report_rows(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)

    if isinstance(obj, str):
        candidate = extract_embedded_json(obj)
        if candidate.startswith('{') or candidate.startswith('['):
            try:
                return find_report_rows(json.loads(candidate), seen)
            except json.JSONDecodeError:
                return None
        return None

    if isinstance(obj, list):
        if obj and isinstance(obj[0], dict) and 'ID Estudiante' in obj[0]:
            return obj
        for item in obj:
            found = find_report_rows(item, seen)
            if found is not None:
                return found
    elif isinstance(obj, dict):
        if obj and 'ID Estudiante' in obj and 'Estudiante' in obj:
            return [obj]
        for value in obj.values():
            found = find_report_rows(value, seen)
            if found is not None:
                return found
    return None

def normalize_scalar(value, column):
    if value is None:
        return ''
    if isinstance(value, bool):
        return 'true' if value else 'false'
    text = str(value).strip()
    if not text:
        return ''
    if column in numeric_cols:
        try:
            numeric = format(Decimal(text), 'f')
            return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
        except (InvalidOperation, ValueError):
            return text
    return text

live_payload = json.loads(live_result_path.read_text(encoding='utf-8'))
live_rows = find_report_rows(live_payload)
if live_rows is None:
    raise RuntimeError('No se pudieron identificar filas del reporte en el resultado SQL')

copy_data_cols = [c for c in copy_rows[0].keys()]
live_by_student = {str(row['ID Estudiante']): row for row in live_rows}
copy_by_student = {str(row['ID Estudiante']): row for row in copy_rows}

all_student_ids = sorted(set(live_by_student) | set(copy_by_student))
post_update_changed = []
post_update_col_counter = Counter()
for student_id in all_student_ids:
    live = live_by_student.get(student_id)
    copy = copy_by_student.get(student_id)
    if live is None or copy is None:
        post_update_changed.append({
            'ID Estudiante': student_id,
            'missing_in': 'live_report' if live is None else 'updated_csv',
        })
        continue
    diffs = []
    for col in copy_data_cols:
        live_value = normalize_scalar(live.get(col), col)
        copy_value = normalize_scalar(copy.get(col), col)
        if live_value != copy_value:
            diffs.append((col, live_value, copy_value))
            post_update_col_counter[col] += 1
    if diffs:
        post_update_changed.append({
            'ID Matrícula': str(copy.get('ID Matrícula', '')),
            'ID Estudiante': student_id,
            'Estudiante': str(copy.get('Estudiante', '')),
            'diffs': diffs,
        })

comparison_status_counts = {
    col: Counter(normalize_scalar(row.get(col), col) or 'VACIO' for row in live_rows)
    for col in compare_cols
}
comparison_issues = []
for row in live_rows:
    issues = {
        col: normalize_scalar(row.get(col), col)
        for col in compare_cols
        if normalize_scalar(row.get(col), col) not in ('OK', '')
    }
    if issues:
        comparison_issues.append({
            'ID Matrícula': str(row.get('ID Matrícula', '')),
            'ID Estudiante': str(row.get('ID Estudiante', '')),
            'Estudiante': str(row.get('Estudiante', '')),
            'issues': issues,
        })

validation_payload = {
    'live_rows': len(live_rows),
    'copy_rows': len(copy_rows),
    'changed_rows_against_updated_csv': len(post_update_changed),
    'column_counts': dict(post_update_col_counter),
    'comparison_status_counts': {k: dict(v) for k, v in comparison_status_counts.items()},
    'comparison_issue_rows': len(comparison_issues),
    'rows': [
        {
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': row['ID Estudiante'],
            'Estudiante': row.get('Estudiante', ''),
            'missing_in': row.get('missing_in'),
            'diffs': [
                {'column': col, 'live_report': old, 'updated_csv': new}
                for col, old, new in row.get('diffs', [])
            ],
        }
        for row in post_update_changed
    ],
    'comparison_issue_details': comparison_issues,
}
validation_json.write_text(json.dumps(validation_payload, ensure_ascii=False, indent=2), encoding='utf-8')

with validation_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID Matrícula', 'ID Estudiante', 'Estudiante', 'column', 'live_report', 'updated_csv'])
    for row in post_update_changed:
        for diff in row.get('diffs', []):
            writer.writerow([
                row.get('ID Matrícula', ''),
                row['ID Estudiante'],
                row.get('Estudiante', ''),
                diff[0],
                diff[1],
                diff[2],
            ])

print('live_rows =', len(live_rows))
print('copy_rows =', len(copy_rows))
print('changed_rows_against_updated_csv =', len(post_update_changed))
print('column_counts =', dict(post_update_col_counter))
print('comparison_issue_rows =', len(comparison_issues))
print('comparison_status_counts =', {k: dict(v) for k, v in comparison_status_counts.items()})
print(validation_json)
print(validation_csv)
for row in post_update_changed[:10]:
    print(f"{row.get('ID Matrícula', '')} | {row['ID Estudiante']} | {row.get('Estudiante', '')}")
    for col, live_value, copy_value in row.get('diffs', []):
        print(f"  - {col}: {live_value!r} -> {copy_value!r}")

live_rows = 432
copy_rows = 432
changed_rows_against_updated_csv = 237
column_counts = {'Matrícula: Monto por Cuota': 220, 'Matrícula: Total Anual': 222, 'Matrícula: N° Cuotas': 169, 'Matrícula: Forma de Pago': 20, 'Matrícula: Prioritario': 5, 'Aranceles: Forma de Pago': 4, 'Aranceles: Total': 1, 'Aranceles: N° Cuota Máxima': 2, 'Aranceles: Monto Cuota Mínimo': 1, 'Aranceles: Monto Cuota Máximo': 1}
comparison_issue_rows = 61
comparison_status_counts = {'Comparación Forma de Pago': {'OK': 425, 'FEE_FALTANTE': 4, 'FORMA_PAGO_DISTINTA': 3}, 'Comparación Cuotas': {'OK': 422, 'CUOTAS_DISTINTAS': 10}, 'Comparación Monto': {'MONTO_DISTINTO': 56, 'OK': 376}, 'Comparación Total': {'TOTAL_DISTINTO': 54, 'OK': 378}}
c:\Meik_Apps\winterhill-cobros-1.1\tmp_post_update_report_validation.json
c:\Meik_Apps\winterhill-cobros-1.1\tmp_post_update_report_validation.csv
3602cbaf-10c5-432a-a94d-142d8eb4abbe | 02fd7937-561a-4a53-ae18-0a3dc8e4517f | ISIDORA ALINE BERROCAL ROA
  - Matrícula: Monto por Cuota: 

In [11]:
original_changed_ids = {row['ID Estudiante'] for row in changed}
post_update_changed_ids = {row['ID Estudiante'] for row in post_update_changed if 'diffs' in row}

remaining_from_original = [row for row in post_update_changed if row.get('ID Estudiante') in original_changed_ids]
newly_observed = [row for row in post_update_changed if row.get('ID Estudiante') not in original_changed_ids]

remaining_original_col_counter = Counter()
for row in remaining_from_original:
    for col, _, _ in row.get('diffs', []):
        remaining_original_col_counter[col] += 1

newly_observed_col_counter = Counter()
for row in newly_observed:
    for col, _, _ in row.get('diffs', []):
        newly_observed_col_counter[col] += 1

print('original_changed_rows =', len(changed))
print('remaining_from_original =', len(remaining_from_original))
print('newly_observed_rows =', len(newly_observed))
print('remaining_from_original_column_counts =', dict(remaining_original_col_counter))
print('newly_observed_column_counts =', dict(newly_observed_col_counter))

original_changed_rows = 83
remaining_from_original = 17
newly_observed_rows = 220
remaining_from_original_column_counts = {'Matrícula: Forma de Pago': 7, 'Matrícula: Prioritario': 5, 'Matrícula: N° Cuotas': 10, 'Matrícula: Monto por Cuota': 10, 'Matrícula: Total Anual': 10, 'Aranceles: Total': 1, 'Aranceles: N° Cuota Máxima': 2, 'Aranceles: Monto Cuota Mínimo': 1, 'Aranceles: Monto Cuota Máximo': 1}
newly_observed_column_counts = {'Matrícula: Monto por Cuota': 210, 'Matrícula: Total Anual': 212, 'Matrícula: N° Cuotas': 159, 'Matrícula: Forma de Pago': 13, 'Aranceles: Forma de Pago': 4}


In [12]:
from collections import defaultdict

new_diff_signature_counter = Counter()
new_column_pair_counter = Counter()
new_amount_pair_counter = Counter()
new_total_pair_counter = Counter()
new_cuotas_pair_counter = Counter()
signature_examples = defaultdict(list)

for row in newly_observed:
    signature = tuple(sorted(col for col, _, _ in row.get('diffs', [])))
    new_diff_signature_counter[signature] += 1
    if len(signature_examples[signature]) < 5:
        signature_examples[signature].append({
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': row.get('ID Estudiante', ''),
            'Estudiante': row.get('Estudiante', ''),
            'diffs': row.get('diffs', []),
        })
    for col, live_value, copy_value in row.get('diffs', []):
        new_column_pair_counter[(col, live_value, copy_value)] += 1
        if col == 'Matrícula: Monto por Cuota':
            new_amount_pair_counter[(live_value, copy_value)] += 1
        elif col == 'Matrícula: Total Anual':
            new_total_pair_counter[(live_value, copy_value)] += 1
        elif col == 'Matrícula: N° Cuotas':
            new_cuotas_pair_counter[(live_value, copy_value)] += 1

print('Top diff signatures:')
for signature, count in new_diff_signature_counter.most_common(15):
    print(count, signature)

print('\nTop monto pairs:')
for pair, count in new_amount_pair_counter.most_common(15):
    print(count, pair)

print('\nTop total pairs:')
for pair, count in new_total_pair_counter.most_common(15):
    print(count, pair)

print('\nTop cuotas pairs:')
for pair, count in new_cuotas_pair_counter.most_common(15):
    print(count, pair)

print('\nExamples for top signatures:')
for signature, count in new_diff_signature_counter.most_common(8):
    print('\nSignature:', signature, 'count=', count)
    for example in signature_examples[signature][:3]:
        print(' ', example['ID Matrícula'], '|', example['ID Estudiante'], '|', example['Estudiante'])
        for diff in example['diffs']:
            print('    ', diff)

Top diff signatures:
155 ('Matrícula: Monto por Cuota', 'Matrícula: N° Cuotas', 'Matrícula: Total Anual')
41 ('Matrícula: Monto por Cuota', 'Matrícula: Total Anual')
10 ('Matrícula: Forma de Pago', 'Matrícula: Monto por Cuota', 'Matrícula: Total Anual')
4 ('Matrícula: Total Anual',)
3 ('Matrícula: Monto por Cuota', 'Matrícula: N° Cuotas')
3 ('Aranceles: Forma de Pago',)
2 ('Matrícula: Forma de Pago',)
1 ('Aranceles: Forma de Pago', 'Matrícula: Monto por Cuota', 'Matrícula: Total Anual')
1 ('Matrícula: Forma de Pago', 'Matrícula: N° Cuotas', 'Matrícula: Total Anual')

Top monto pairs:
155 ('', '0')
7 ('25718', '102870')
7 ('61722', '102870')
6 ('79876', '133126')
5 ('93188', '133126')
4 ('36005', '102870')
3 ('72009', '102870')
3 ('92583', '102870')
3 ('53250', '133126')
2 ('41148', '102870')
2 ('205740', '102870')
2 ('106501', '133126')
2 ('51435', '102870')
1 ('86532', '133126')
1 ('266252', '133126')

Top total pairs:
155 ('', '0')
7 ('257180', '1028700')
7 ('617220', '1028700')
6 ('

In [13]:
audit_summary_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_summary.json')

audit_summary = {
    'newly_observed_rows': len(newly_observed),
    'top_diff_signatures': [
        {
            'count': count,
            'signature': list(signature),
            'examples': signature_examples[signature][:3],
        }
        for signature, count in new_diff_signature_counter.most_common(15)
    ],
    'top_monto_pairs': [
        {'count': count, 'live_report': pair[0], 'updated_csv': pair[1]}
        for pair, count in new_amount_pair_counter.most_common(15)
    ],
    'top_total_pairs': [
        {'count': count, 'live_report': pair[0], 'updated_csv': pair[1]}
        for pair, count in new_total_pair_counter.most_common(15)
    ],
    'top_cuotas_pairs': [
        {'count': count, 'live_report': pair[0], 'updated_csv': pair[1]}
        for pair, count in new_cuotas_pair_counter.most_common(15)
    ],
    'top_column_pairs': [
        {'count': count, 'column': col, 'live_report': live_value, 'updated_csv': copy_value}
        for (col, live_value, copy_value), count in new_column_pair_counter.most_common(25)
    ],
}

audit_summary_path.write_text(json.dumps(audit_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(audit_summary_path)

c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_summary.json


In [14]:
audit_detail_json = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_detail.json')
audit_detail_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_detail.csv')

def classify_new_difference(row):
    diffs = {col: (live_value, copy_value) for col, live_value, copy_value in row.get('diffs', [])}
    columns = tuple(sorted(diffs))

    if columns == ('Matrícula: Monto por Cuota', 'Matrícula: N° Cuotas', 'Matrícula: Total Anual') and all(diffs[col] == ('', '0') for col in columns):
        return (
            'matricula_blank_vs_zero',
            'Campos de matrícula vacíos en BD/reporte y explícitamente en 0 en el CSV; parece normalización pendiente, no cambio financiero material.'
        )

    if columns == ('Matrícula: Monto por Cuota', 'Matrícula: Total Anual'):
        return (
            'matricula_amount_total_rebased',
            'Montos de matrícula en BD quedan por debajo del arancel objetivo del CSV; patrón masivo de rebasing a tarifa anual estándar.'
        )

    if columns == ('Matrícula: Forma de Pago', 'Matrícula: Monto por Cuota', 'Matrícula: Total Anual'):
        return (
            'matricula_method_and_amount_rebased',
            'Además del rebasing de montos, cambia la forma de pago declarada en matrícula; requiere confirmar fuente de verdad.'
        )

    if columns == ('Matrícula: Total Anual',):
        live_total, copy_total = diffs['Matrícula: Total Anual']
        try:
            delta = abs(Decimal(live_total) - Decimal(copy_total))
        except Exception:
            delta = None
        if delta is not None and delta <= Decimal('3'):
            return (
                'matricula_minor_rounding_total',
                'Diferencia mínima de redondeo o ajuste manual fino en el total anual.'
            )
        return (
            'matricula_total_only',
            'Solo cambia el total anual de matrícula; revisar cálculo o ajuste puntual.'
        )

    if columns == ('Matrícula: Monto por Cuota', 'Matrícula: N° Cuotas'):
        return (
            'matricula_installment_replan',
            'Replanificación de cuotas en matrícula sin ajuste explícito del total; cambio estructural del plan.'
        )

    if columns == ('Aranceles: Forma de Pago',):
        return (
            'fee_mixed_payment_method',
            'En aranceles la BD combina formas de pago y el CSV espera una sola; probable residuo de pagos históricos o mezcla no limpiada.'
        )

    if columns == ('Matrícula: Forma de Pago',):
        return (
            'matricula_payment_method_only',
            'Solo difiere la forma de pago de matrícula; revisar meta.payment_method sin tocar montos.'
        )

    if columns == ('Aranceles: Forma de Pago', 'Matrícula: Monto por Cuota', 'Matrícula: Total Anual'):
        return (
            'mixed_fee_method_and_amount_rebased',
            'Se combinan rebasing de matrícula y mezcla de formas de pago en aranceles.'
        )

    if columns == ('Matrícula: Forma de Pago', 'Matrícula: N° Cuotas', 'Matrícula: Total Anual'):
        return (
            'matricula_method_replan_total',
            'Cambio simultáneo de forma de pago, número de cuotas y total anual; caso singular que requiere revisión individual.'
        )

    return (
        'other',
        'Patrón no clasificado automáticamente; revisar individualmente.'
    )

audit_rows = []
audit_category_counter = Counter()
for row in newly_observed:
    category, probable_cause = classify_new_difference(row)
    audit_category_counter[category] += 1
    audit_rows.append({
        'category': category,
        'probable_cause': probable_cause,
        'ID Matrícula': row.get('ID Matrícula', ''),
        'ID Estudiante': row.get('ID Estudiante', ''),
        'Estudiante': row.get('Estudiante', ''),
        'diff_count': len(row.get('diffs', [])),
        'diffs': [
            {'column': col, 'live_report': live_value, 'updated_csv': copy_value}
            for col, live_value, copy_value in row.get('diffs', [])
        ],
    })

audit_detail_json.write_text(
    json.dumps({
        'rows': audit_rows,
        'category_counts': dict(audit_category_counter),
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

with audit_detail_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['category', 'probable_cause', 'ID Matrícula', 'ID Estudiante', 'Estudiante', 'column', 'live_report', 'updated_csv'])
    for row in audit_rows:
        for diff in row['diffs']:
            writer.writerow([
                row['category'],
                row['probable_cause'],
                row['ID Matrícula'],
                row['ID Estudiante'],
                row['Estudiante'],
                diff['column'],
                diff['live_report'],
                diff['updated_csv'],
            ])

print('category_counts =', dict(audit_category_counter))
print(audit_detail_json)
print(audit_detail_csv)

category_counts = {'matricula_amount_total_rebased': 41, 'matricula_blank_vs_zero': 155, 'matricula_method_and_amount_rebased': 10, 'matricula_installment_replan': 3, 'fee_mixed_payment_method': 3, 'mixed_fee_method_and_amount_rebased': 1, 'matricula_minor_rounding_total': 4, 'matricula_payment_method_only': 2, 'matricula_method_replan_total': 1}
c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_detail.json
c:\Meik_Apps\winterhill-cobros-1.1\tmp_newly_observed_audit_detail.csv


In [6]:
csv_diff_json = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.json')
csv_diff_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.csv')

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip()

def normalize_decimal_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        numeric = format(Decimal(text), 'f')
        return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
    except (InvalidOperation, ValueError):
        return text

def normalize_int_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, ValueError):
        return text

def split_payment_methods(value):
    text = normalize_text(value).upper()
    if not text:
        return []
    return [part.strip() for part in text.split('|') if part.strip()]

csv_internal_diffs = []
csv_internal_counter = Counter()
for row in copy_rows:
    matricula_method = normalize_text(row.get('Matrícula: Forma de Pago')).upper()
    arancel_method = normalize_text(row.get('Aranceles: Forma de Pago')).upper()
    matricula_prioritario = normalize_text(row.get('Matrícula: Prioritario')).lower() == 'true'
    matricula_cuotas = normalize_int_text(row.get('Matrícula: N° Cuotas'))
    arancel_cuotas = normalize_int_text(row.get('Aranceles: N° Cuotas'))
    matricula_monto = normalize_decimal_text(row.get('Matrícula: Monto por Cuota'))
    arancel_min = normalize_decimal_text(row.get('Aranceles: Monto Cuota Mínimo'))
    arancel_max = normalize_decimal_text(row.get('Aranceles: Monto Cuota Máximo'))
    matricula_total = normalize_decimal_text(row.get('Matrícula: Total Anual'))
    arancel_total = normalize_decimal_text(row.get('Aranceles: Total'))

    diffs = []

    if matricula_method == 'PRIORITARIO':
        if arancel_cuotas not in ('', '0'):
            diffs.append({
                'field': 'Forma de Pago',
                'issue': 'PRIORITARIO_CON_FEE',
                'matricula': matricula_method,
                'aranceles': arancel_method or 'SIN_FEE',
            })
    else:
        arancel_methods = split_payment_methods(arancel_method)
        if arancel_method == 'SIN_FEE' or arancel_cuotas in ('', '0'):
            if matricula_cuotas not in ('', '0'):
                diffs.append({
                    'field': 'Forma de Pago',
                    'issue': 'FEE_FALTANTE',
                    'matricula': matricula_method,
                    'aranceles': arancel_method or 'SIN_FEE',
                })
        elif matricula_method and matricula_method not in arancel_methods:
            diffs.append({
                'field': 'Forma de Pago',
                'issue': 'FORMA_PAGO_DISTINTA',
                'matricula': matricula_method,
                'aranceles': arancel_method,
            })

    if matricula_cuotas != arancel_cuotas:
        diffs.append({
            'field': 'Cuotas',
            'issue': 'CUOTAS_DISTINTAS',
            'matricula': matricula_cuotas,
            'aranceles': arancel_cuotas,
        })

    if matricula_monto != arancel_min or matricula_monto != arancel_max:
        diffs.append({
            'field': 'Monto',
            'issue': 'MONTO_DISTINTO',
            'matricula': matricula_monto,
            'aranceles': f'{arancel_min}..{arancel_max}',
        })

    if matricula_total != arancel_total:
        diffs.append({
            'field': 'Total',
            'issue': 'TOTAL_DISTINTO',
            'matricula': matricula_total,
            'aranceles': arancel_total,
        })

    if diffs:
        csv_internal_diffs.append({
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': row.get('ID Estudiante', ''),
            'RUT': row.get('RUT', ''),
            'Estudiante': row.get('Estudiante', ''),
            'Curso': row.get('Curso', ''),
            'Estado Matrícula': row.get('Estado Matrícula', ''),
            'diffs': diffs,
        })
        for diff in diffs:
            csv_internal_counter[diff['issue']] += 1

csv_diff_payload = {
    'rows_with_differences': len(csv_internal_diffs),
    'issue_counts': dict(csv_internal_counter),
    'rows': csv_internal_diffs,
}
csv_diff_json.write_text(json.dumps(csv_diff_payload, ensure_ascii=False, indent=2), encoding='utf-8')

with csv_diff_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID Matrícula', 'ID Estudiante', 'RUT', 'Estudiante', 'Curso', 'Estado Matrícula', 'field', 'issue', 'matricula', 'aranceles'])
    for row in csv_internal_diffs:
        for diff in row['diffs']:
            writer.writerow([
                row['ID Matrícula'],
                row['ID Estudiante'],
                row['RUT'],
                row['Estudiante'],
                row['Curso'],
                row['Estado Matrícula'],
                diff['field'],
                diff['issue'],
                diff['matricula'],
                diff['aranceles'],
            ])

print('rows_with_differences =', len(csv_internal_diffs))
print('issue_counts =', dict(csv_internal_counter))
print(csv_diff_json)
print(csv_diff_csv)
for row in csv_internal_diffs[:20]:
    print(f"{row['ID Matrícula']} | {row['ID Estudiante']} | {row['Estudiante']} | {row['Curso']}")
    for diff in row['diffs']:
        print(f"  - {diff['issue']}: {diff['field']} | matrícula={diff['matricula']!r} | aranceles={diff['aranceles']!r}")

NameError: name 'copy_rows' is not defined

In [16]:
csv_diff_records_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv')

with csv_diff_records_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'ID Matrícula',
        'ID Estudiante',
        'RUT',
        'Estudiante',
        'Curso',
        'Estado Matrícula',
        'issues',
        'matricula_forma_pago',
        'aranceles_forma_pago',
        'matricula_cuotas',
        'aranceles_cuotas',
        'matricula_monto',
        'aranceles_monto_min',
        'aranceles_monto_max',
        'matricula_total',
        'aranceles_total',
    ])
    for row in csv_internal_diffs:
        original = next(item for item in copy_rows if item['ID Estudiante'] == row['ID Estudiante'])
        writer.writerow([
            row['ID Matrícula'],
            row['ID Estudiante'],
            row['RUT'],
            row['Estudiante'],
            row['Curso'],
            row['Estado Matrícula'],
            '|'.join(diff['issue'] for diff in row['diffs']),
            original.get('Matrícula: Forma de Pago', ''),
            original.get('Aranceles: Forma de Pago', ''),
            original.get('Matrícula: N° Cuotas', ''),
            original.get('Aranceles: N° Cuotas', ''),
            original.get('Matrícula: Monto por Cuota', ''),
            original.get('Aranceles: Monto Cuota Mínimo', ''),
            original.get('Aranceles: Monto Cuota Máximo', ''),
            original.get('Matrícula: Total Anual', ''),
            original.get('Aranceles: Total', ''),
        ])

print(csv_diff_records_csv)

c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv


In [4]:
import csv
import json
import re
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
live_result_path = Path(r'c:\Users\eduar\AppData\Roaming\Code\User\workspaceStorage\220c91a4418e7742dc4af0311d3a2b08\GitHub.copilot-chat\chat-session-resources\2a8d9d80-8e33-42e8-8041-6ae5e718bc8c\call_UgT1sWUuSBKgu9V6paqiL43r__vscode-1775828510350\content.json')

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    local_copy_rows = list(csv.DictReader(f))
copy_data_cols = list(local_copy_rows[0].keys())

def extract_embedded_json(text):
    candidate = text.strip()
    match = re.search(r'<untrusted-data[^>]*>\s*(\[.*\]|\{.*\})\s*</untrusted-data[^>]*>', candidate, re.S)
    if match:
        return match.group(1)
    bracket_start = candidate.find('[')
    bracket_end = candidate.rfind(']')
    if bracket_start != -1 and bracket_end != -1 and bracket_end > bracket_start:
        return candidate[bracket_start:bracket_end + 1]
    brace_start = candidate.find('{')
    brace_end = candidate.rfind('}')
    if brace_start != -1 and brace_end != -1 and brace_end > brace_start:
        return candidate[brace_start:brace_end + 1]
    return candidate

def find_report_rows(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)
    if isinstance(obj, str):
        candidate = extract_embedded_json(obj)
        if candidate.startswith('{') or candidate.startswith('['):
            try:
                return find_report_rows(json.loads(candidate), seen)
            except json.JSONDecodeError:
                return None
        return None
    if isinstance(obj, list):
        if obj and isinstance(obj[0], dict) and 'ID Estudiante' in obj[0]:
            return obj
        for item in obj:
            found = find_report_rows(item, seen)
            if found is not None:
                return found
    elif isinstance(obj, dict):
        if obj and 'ID Estudiante' in obj and 'Estudiante' in obj:
            return [obj]
        for value in obj.values():
            found = find_report_rows(value, seen)
            if found is not None:
                return found
    return None

numeric_cols = {
    'Matrícula: N° Cuotas',
    'Matrícula: Monto por Cuota',
    'Matrícula: Total Anual',
    'Aranceles: N° Cuotas',
    'Aranceles: Monto Cuota Mínimo',
    'Aranceles: Monto Cuota Máximo',
    'Aranceles: Total',
    'Aranceles: N° Cuota Mínima',
    'Aranceles: N° Cuota Máxima',
}

def normalize_text(value):
    return '' if value is None else str(value).strip()

def normalize_scalar(value, column):
    text = normalize_text(value)
    if not text:
        return ''
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if column in numeric_cols:
        try:
            numeric = format(Decimal(text), 'f')
            return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
        except (InvalidOperation, ValueError):
            return text
    return text

def normalize_decimal_text(value):
    return normalize_scalar(value, 'Matrícula: Monto por Cuota')

def normalize_int_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, ValueError):
        return text

def split_payment_methods(value):
    text = normalize_text(value).upper()
    if not text:
        return []
    return [part.strip() for part in text.split('|') if part.strip()]

live_payload = json.loads(live_result_path.read_text(encoding='utf-8'))
live_rows = find_report_rows(live_payload)
live_by_student = {str(row['ID Estudiante']): row for row in live_rows}

local_csv_internal_diffs = []
for row in local_copy_rows:
    matricula_method = normalize_text(row.get('Matrícula: Forma de Pago')).upper()
    arancel_method = normalize_text(row.get('Aranceles: Forma de Pago')).upper()
    matricula_cuotas = normalize_int_text(row.get('Matrícula: N° Cuotas'))
    arancel_cuotas = normalize_int_text(row.get('Aranceles: N° Cuotas'))
    matricula_monto = normalize_decimal_text(row.get('Matrícula: Monto por Cuota'))
    arancel_min = normalize_scalar(row.get('Aranceles: Monto Cuota Mínimo'), 'Aranceles: Monto Cuota Mínimo')
    arancel_max = normalize_scalar(row.get('Aranceles: Monto Cuota Máximo'), 'Aranceles: Monto Cuota Máximo')
    matricula_total = normalize_scalar(row.get('Matrícula: Total Anual'), 'Matrícula: Total Anual')
    arancel_total = normalize_scalar(row.get('Aranceles: Total'), 'Aranceles: Total')
    diffs = []
    if matricula_method == 'PRIORITARIO':
        if arancel_cuotas not in ('', '0'):
            diffs.append('PRIORITARIO_CON_FEE')
    else:
        arancel_methods = split_payment_methods(arancel_method)
        if arancel_method == 'SIN_FEE' or arancel_cuotas in ('', '0'):
            if matricula_cuotas not in ('', '0'):
                diffs.append('FEE_FALTANTE')
        elif matricula_method and matricula_method not in arancel_methods:
            diffs.append('FORMA_PAGO_DISTINTA')
    if matricula_cuotas != arancel_cuotas:
        diffs.append('CUOTAS_DISTINTAS')
    if matricula_monto != arancel_min or matricula_monto != arancel_max:
        diffs.append('MONTO_DISTINTO')
    if matricula_total != arancel_total:
        diffs.append('TOTAL_DISTINTO')
    if diffs:
        local_csv_internal_diffs.append(row)

problem_student_ids = {row['ID Estudiante'] for row in local_csv_internal_diffs}
db_sync_candidates = []
db_sync_col_counter = Counter()
for row in local_copy_rows:
    student_id = row['ID Estudiante']
    if student_id not in problem_student_ids:
        continue
    live = live_by_student.get(student_id)
    if not live:
        continue
    diffs = []
    for col in copy_data_cols:
        copy_value = normalize_scalar(row.get(col), col)
        live_value = normalize_scalar(live.get(col), col)
        if copy_value != live_value:
            diffs.append((col, copy_value, live_value))
            db_sync_col_counter[col] += 1
    if diffs:
        db_sync_candidates.append({
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': student_id,
            'Estudiante': row.get('Estudiante', ''),
            'diffs': diffs,
        })

print('problem_rows_in_csv =', len(problem_student_ids))
print('rows_needing_db_sync =', len(db_sync_candidates))
print('db_sync_column_counts =', dict(db_sync_col_counter))
for row in db_sync_candidates[:10]:
    print(f"{row['ID Matrícula']} | {row['ID Estudiante']} | {row['Estudiante']}")
    for col, copy_value, live_value in row['diffs']:
        print(f"  - {col}: {copy_value!r} -> {live_value!r}")

problem_rows_in_csv = 218
rows_needing_db_sync = 0
db_sync_column_counts = {}


In [3]:
import csv
import json
import re
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
live_result_path = Path(r'c:\Users\eduar\AppData\Roaming\Code\User\workspaceStorage\220c91a4418e7742dc4af0311d3a2b08\GitHub.copilot-chat\chat-session-resources\2a8d9d80-8e33-42e8-8041-6ae5e718bc8c\call_UgT1sWUuSBKgu9V6paqiL43r__vscode-1775828510350\content.json')
sync_audit_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_problem_rows_synced_from_db.json')

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
    headers = list(rows[0].keys())

def extract_embedded_json(text):
    candidate = text.strip()
    match = re.search(r'<untrusted-data[^>]*>\s*(\[.*\]|\{.*\})\s*</untrusted-data[^>]*>', candidate, re.S)
    if match:
        return match.group(1)
    bracket_start = candidate.find('[')
    bracket_end = candidate.rfind(']')
    if bracket_start != -1 and bracket_end != -1 and bracket_end > bracket_start:
        return candidate[bracket_start:bracket_end + 1]
    brace_start = candidate.find('{')
    brace_end = candidate.rfind('}')
    if brace_start != -1 and brace_end != -1 and brace_end > brace_start:
        return candidate[brace_start:brace_end + 1]
    return candidate

def find_report_rows(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return None
    seen.add(obj_id)
    if isinstance(obj, str):
        candidate = extract_embedded_json(obj)
        if candidate.startswith('{') or candidate.startswith('['):
            try:
                return find_report_rows(json.loads(candidate), seen)
            except json.JSONDecodeError:
                return None
        return None
    if isinstance(obj, list):
        if obj and isinstance(obj[0], dict) and 'ID Estudiante' in obj[0]:
            return obj
        for item in obj:
            found = find_report_rows(item, seen)
            if found is not None:
                return found
    elif isinstance(obj, dict):
        if obj and 'ID Estudiante' in obj and 'Estudiante' in obj:
            return [obj]
        for value in obj.values():
            found = find_report_rows(value, seen)
            if found is not None:
                return found
    return None

numeric_cols = {
    'Matrícula: N° Cuotas',
    'Matrícula: Monto por Cuota',
    'Matrícula: Total Anual',
    'Aranceles: N° Cuotas',
    'Aranceles: Monto Cuota Mínimo',
    'Aranceles: Monto Cuota Máximo',
    'Aranceles: Total',
    'Aranceles: N° Cuota Mínima',
    'Aranceles: N° Cuota Máxima',
}

def as_csv_text(value):
    if value is None:
        return ''
    if isinstance(value, bool):
        return 'true' if value else 'false'
    return str(value)

def normalize_text(value):
    return '' if value is None else str(value).strip()

def normalize_scalar(value, column):
    if isinstance(value, bool):
        return 'true' if value else 'false'
    text = normalize_text(value)
    if not text:
        return ''
    if column in numeric_cols:
        try:
            numeric = format(Decimal(text), 'f')
            return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
        except (InvalidOperation, ValueError):
            return text
    return text

def normalize_int_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, ValueError):
        return text

def split_payment_methods(value):
    text = normalize_text(value).upper()
    if not text:
        return []
    return [part.strip() for part in text.split('|') if part.strip()]

live_payload = json.loads(live_result_path.read_text(encoding='utf-8'))
live_rows = find_report_rows(live_payload)
live_by_student = {str(row['ID Estudiante']): row for row in live_rows}

problem_student_ids = set()
for row in rows:
    matricula_method = normalize_text(row.get('Matrícula: Forma de Pago')).upper()
    arancel_method = normalize_text(row.get('Aranceles: Forma de Pago')).upper()
    matricula_cuotas = normalize_int_text(row.get('Matrícula: N° Cuotas'))
    arancel_cuotas = normalize_int_text(row.get('Aranceles: N° Cuotas'))
    matricula_monto = normalize_scalar(row.get('Matrícula: Monto por Cuota'), 'Matrícula: Monto por Cuota')
    arancel_min = normalize_scalar(row.get('Aranceles: Monto Cuota Mínimo'), 'Aranceles: Monto Cuota Mínimo')
    arancel_max = normalize_scalar(row.get('Aranceles: Monto Cuota Máximo'), 'Aranceles: Monto Cuota Máximo')
    matricula_total = normalize_scalar(row.get('Matrícula: Total Anual'), 'Matrícula: Total Anual')
    arancel_total = normalize_scalar(row.get('Aranceles: Total'), 'Aranceles: Total')
    has_diff = False
    if matricula_method == 'PRIORITARIO':
        has_diff = arancel_cuotas not in ('', '0')
    else:
        arancel_methods = split_payment_methods(arancel_method)
        if arancel_method == 'SIN_FEE' or arancel_cuotas in ('', '0'):
            has_diff = matricula_cuotas not in ('', '0')
        elif matricula_method and matricula_method not in arancel_methods:
            has_diff = True
    has_diff = has_diff or (matricula_cuotas != arancel_cuotas)
    has_diff = has_diff or (matricula_monto != arancel_min or matricula_monto != arancel_max)
    has_diff = has_diff or (matricula_total != arancel_total)
    if has_diff:
        problem_student_ids.add(row['ID Estudiante'])

updated_rows = []
audit_rows = []
changed_row_count = 0
changed_cell_count = 0
changed_columns = Counter()

for row in rows:
    student_id = row['ID Estudiante']
    live = live_by_student.get(student_id)
    updated = dict(row)
    row_changed = False
    if student_id in problem_student_ids and live is not None:
        for col in headers:
            copy_value = normalize_scalar(row.get(col), col)
            live_value = normalize_scalar(live.get(col), col)
            if copy_value != live_value:
                updated[col] = as_csv_text(live.get(col))
                row_changed = True
                changed_cell_count += 1
                changed_columns[col] += 1
                audit_rows.append({
                    'ID Matrícula': row.get('ID Matrícula', ''),
                    'ID Estudiante': student_id,
                    'Estudiante': row.get('Estudiante', ''),
                    'column': col,
                    'old_value': row.get(col, ''),
                    'new_value': updated[col],
                })
    if row_changed:
        changed_row_count += 1
    updated_rows.append(updated)

with copy_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(updated_rows)

sync_audit_path.write_text(
    json.dumps({
        'changed_row_count': changed_row_count,
        'changed_cell_count': changed_cell_count,
        'changed_columns': dict(changed_columns),
        'changes': audit_rows,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('changed_row_count =', changed_row_count)
print('changed_cell_count =', changed_cell_count)
print('changed_columns =', dict(changed_columns))
print(sync_audit_path)

changed_row_count = 230
changed_cell_count = 635
changed_columns = {'Matrícula: Monto por Cuota': 219, 'Matrícula: Total Anual': 221, 'Matrícula: Forma de Pago': 17, 'Matrícula: N° Cuotas': 168, 'Matrícula: Prioritario': 4, 'Aranceles: Monto Cuota Mínimo': 1, 'Aranceles: Monto Cuota Máximo': 1, 'Aranceles: N° Cuota Máxima': 2, 'Aranceles: Forma de Pago': 1, 'Aranceles: Total': 1}
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_problem_rows_synced_from_db.json


In [5]:
import csv
import json
from pathlib import Path

copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
fill_zero_audit_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_zero_matricula_filled_aranceles.json')

target_arancel_cols = [
    'Aranceles: Monto Cuota Mínimo',
    'Aranceles: Monto Cuota Máximo',
    'Aranceles: Total',
    'Aranceles: N° Cuota Mínima',
    'Aranceles: N° Cuota Máxima',
]

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))
    headers = list(rows[0].keys())

updated_rows = []
audit_rows = []
target_row_count = 0
changed_cell_count = 0

for row in rows:
    updated = dict(row)
    is_zero_matricula = (
        str(row.get('Matrícula: N° Cuotas', '')).strip() == '0'
        and str(row.get('Matrícula: Monto por Cuota', '')).strip() == '0'
        and str(row.get('Matrícula: Total Anual', '')).strip() == '0'
    )
    has_blank_arancel = any(str(row.get(col, '')).strip() == '' for col in target_arancel_cols)

    if is_zero_matricula and has_blank_arancel:
        target_row_count += 1
        for col in target_arancel_cols:
            old_value = str(row.get(col, ''))
            if old_value.strip() == '':
                updated[col] = '0'
                changed_cell_count += 1
                audit_rows.append({
                    'ID Matrícula': row.get('ID Matrícula', ''),
                    'ID Estudiante': row.get('ID Estudiante', ''),
                    'Estudiante': row.get('Estudiante', ''),
                    'column': col,
                    'old_value': old_value,
                    'new_value': '0',
                })

    updated_rows.append(updated)

with copy_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(updated_rows)

fill_zero_audit_path.write_text(
    json.dumps({
        'target_row_count': target_row_count,
        'changed_cell_count': changed_cell_count,
        'changes': audit_rows,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('target_row_count =', target_row_count)
print('changed_cell_count =', changed_cell_count)
print(fill_zero_audit_path)

target_row_count = 46
changed_cell_count = 230
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_zero_matricula_filled_aranceles.json


In [7]:
import csv
import json
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
post_fill_validation_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_post_fill_validation.json')

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip()

def normalize_decimal_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        numeric = format(Decimal(text), 'f')
        return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
    except (InvalidOperation, ValueError):
        return text

def normalize_int_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, ValueError):
        return text

def split_payment_methods(value):
    text = normalize_text(value).upper()
    if not text:
        return []
    return [part.strip() for part in text.split('|') if part.strip()]

csv_internal_diffs = []
csv_internal_counter = Counter()
for row in rows:
    matricula_method = normalize_text(row.get('Matrícula: Forma de Pago')).upper()
    arancel_method = normalize_text(row.get('Aranceles: Forma de Pago')).upper()
    matricula_cuotas = normalize_int_text(row.get('Matrícula: N° Cuotas'))
    arancel_cuotas = normalize_int_text(row.get('Aranceles: N° Cuotas'))
    matricula_monto = normalize_decimal_text(row.get('Matrícula: Monto por Cuota'))
    arancel_min = normalize_decimal_text(row.get('Aranceles: Monto Cuota Mínimo'))
    arancel_max = normalize_decimal_text(row.get('Aranceles: Monto Cuota Máximo'))
    matricula_total = normalize_decimal_text(row.get('Matrícula: Total Anual'))
    arancel_total = normalize_decimal_text(row.get('Aranceles: Total'))

    diffs = []
    if matricula_method == 'PRIORITARIO':
        if arancel_cuotas not in ('', '0'):
            diffs.append('PRIORITARIO_CON_FEE')
    else:
        arancel_methods = split_payment_methods(arancel_method)
        if arancel_method == 'SIN_FEE' or arancel_cuotas in ('', '0'):
            if matricula_cuotas not in ('', '0'):
                diffs.append('FEE_FALTANTE')
        elif matricula_method and matricula_method not in arancel_methods:
            diffs.append('FORMA_PAGO_DISTINTA')
    if matricula_cuotas != arancel_cuotas:
        diffs.append('CUOTAS_DISTINTAS')
    if matricula_monto != arancel_min or matricula_monto != arancel_max:
        diffs.append('MONTO_DISTINTO')
    if matricula_total != arancel_total:
        diffs.append('TOTAL_DISTINTO')
    if diffs:
        csv_internal_diffs.append({
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': row.get('ID Estudiante', ''),
            'Estudiante': row.get('Estudiante', ''),
            'issues': diffs,
        })
        for diff in diffs:
            csv_internal_counter[diff] += 1

post_fill_validation_path.write_text(
    json.dumps({
        'rows_with_differences': len(csv_internal_diffs),
        'issue_counts': dict(csv_internal_counter),
        'rows': csv_internal_diffs,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('rows_with_differences =', len(csv_internal_diffs))
print('issue_counts =', dict(csv_internal_counter))
print(post_fill_validation_path)

rows_with_differences = 172
issue_counts = {'CUOTAS_DISTINTAS': 169, 'MONTO_DISTINTO': 9, 'FEE_FALTANTE': 4, 'TOTAL_DISTINTO': 7}
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_post_fill_validation.json


In [8]:
import csv
import json
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

copy_path = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
csv_diff_json = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.json')
csv_diff_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.csv')
csv_diff_records_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv')

with copy_path.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip()

def normalize_decimal_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        numeric = format(Decimal(text), 'f')
        return numeric.rstrip('0').rstrip('.') if '.' in numeric else numeric
    except (InvalidOperation, ValueError):
        return text

def normalize_int_text(value):
    text = normalize_text(value)
    if not text:
        return ''
    try:
        return str(int(Decimal(text)))
    except (InvalidOperation, ValueError):
        return text

def split_payment_methods(value):
    text = normalize_text(value).upper()
    if not text:
        return []
    return [part.strip() for part in text.split('|') if part.strip()]

def is_zeroish(value):
    text = normalize_text(value)
    if not text:
        return True
    try:
        return Decimal(text) == 0
    except (InvalidOperation, ValueError):
        return False

csv_internal_diffs = []
csv_internal_counter = Counter()

for row in rows:
    matricula_method = normalize_text(row.get('Matrícula: Forma de Pago')).upper()
    arancel_method = normalize_text(row.get('Aranceles: Forma de Pago')).upper()
    matricula_cuotas = normalize_int_text(row.get('Matrícula: N° Cuotas'))
    arancel_cuotas = normalize_int_text(row.get('Aranceles: N° Cuotas'))
    matricula_monto = normalize_decimal_text(row.get('Matrícula: Monto por Cuota'))
    arancel_min = normalize_decimal_text(row.get('Aranceles: Monto Cuota Mínimo'))
    arancel_max = normalize_decimal_text(row.get('Aranceles: Monto Cuota Máximo'))
    matricula_total = normalize_decimal_text(row.get('Matrícula: Total Anual'))
    arancel_total = normalize_decimal_text(row.get('Aranceles: Total'))

    zero_equivalent_case = (
        matricula_method == 'PRIORITARIO' or arancel_method == 'SIN_FEE'
    )

    diffs = []

    if matricula_method == 'PRIORITARIO':
        if not is_zeroish(arancel_cuotas):
            diffs.append({
                'field': 'Forma de Pago',
                'issue': 'PRIORITARIO_CON_FEE',
                'matricula': matricula_method,
                'aranceles': arancel_method or 'SIN_FEE',
            })
    else:
        arancel_methods = split_payment_methods(arancel_method)
        if arancel_method == 'SIN_FEE' or is_zeroish(arancel_cuotas):
            if not is_zeroish(matricula_cuotas):
                diffs.append({
                    'field': 'Forma de Pago',
                    'issue': 'FEE_FALTANTE',
                    'matricula': matricula_method,
                    'aranceles': arancel_method or 'SIN_FEE',
                })
        elif matricula_method and matricula_method not in arancel_methods:
            diffs.append({
                'field': 'Forma de Pago',
                'issue': 'FORMA_PAGO_DISTINTA',
                'matricula': matricula_method,
                'aranceles': arancel_method,
            })

    cuotas_equal = matricula_cuotas == arancel_cuotas or (zero_equivalent_case and is_zeroish(matricula_cuotas) and is_zeroish(arancel_cuotas))
    monto_equal = (
        matricula_monto == arancel_min == arancel_max
        or (zero_equivalent_case and is_zeroish(matricula_monto) and is_zeroish(arancel_min) and is_zeroish(arancel_max))
    )
    total_equal = matricula_total == arancel_total or (zero_equivalent_case and is_zeroish(matricula_total) and is_zeroish(arancel_total))

    if not cuotas_equal:
        diffs.append({
            'field': 'Cuotas',
            'issue': 'CUOTAS_DISTINTAS',
            'matricula': matricula_cuotas,
            'aranceles': arancel_cuotas,
        })

    if not monto_equal:
        diffs.append({
            'field': 'Monto',
            'issue': 'MONTO_DISTINTO',
            'matricula': matricula_monto,
            'aranceles': f'{arancel_min}..{arancel_max}',
        })

    if not total_equal:
        diffs.append({
            'field': 'Total',
            'issue': 'TOTAL_DISTINTO',
            'matricula': matricula_total,
            'aranceles': arancel_total,
        })

    if diffs:
        csv_internal_diffs.append({
            'ID Matrícula': row.get('ID Matrícula', ''),
            'ID Estudiante': row.get('ID Estudiante', ''),
            'RUT': row.get('RUT', ''),
            'Estudiante': row.get('Estudiante', ''),
            'Curso': row.get('Curso', ''),
            'Estado Matrícula': row.get('Estado Matrícula', ''),
            'diffs': diffs,
        })
        for diff in diffs:
            csv_internal_counter[diff['issue']] += 1

csv_diff_payload = {
    'rows_with_differences': len(csv_internal_diffs),
    'issue_counts': dict(csv_internal_counter),
    'equivalence_rule': '0 en matrícula se considera equivalente a vacío/NULL/0 en aranceles para casos PRIORITARIO o SIN_FEE',
    'rows': csv_internal_diffs,
}
csv_diff_json.write_text(json.dumps(csv_diff_payload, ensure_ascii=False, indent=2), encoding='utf-8')

with csv_diff_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID Matrícula', 'ID Estudiante', 'RUT', 'Estudiante', 'Curso', 'Estado Matrícula', 'field', 'issue', 'matricula', 'aranceles'])
    for row in csv_internal_diffs:
        for diff in row['diffs']:
            writer.writerow([
                row['ID Matrícula'],
                row['ID Estudiante'],
                row['RUT'],
                row['Estudiante'],
                row['Curso'],
                row['Estado Matrícula'],
                diff['field'],
                diff['issue'],
                diff['matricula'],
                diff['aranceles'],
            ])

with csv_diff_records_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'ID Matrícula',
        'ID Estudiante',
        'RUT',
        'Estudiante',
        'Curso',
        'Estado Matrícula',
        'issues',
        'matricula_forma_pago',
        'aranceles_forma_pago',
        'matricula_cuotas',
        'aranceles_cuotas',
        'matricula_monto',
        'aranceles_monto_min',
        'aranceles_monto_max',
        'matricula_total',
        'aranceles_total',
    ])
    for row in csv_internal_diffs:
        original = next(item for item in rows if item['ID Estudiante'] == row['ID Estudiante'])
        writer.writerow([
            row['ID Matrícula'],
            row['ID Estudiante'],
            row['RUT'],
            row['Estudiante'],
            row['Curso'],
            row['Estado Matrícula'],
            '|'.join(diff['issue'] for diff in row['diffs']),
            original.get('Matrícula: Forma de Pago', ''),
            original.get('Aranceles: Forma de Pago', ''),
            original.get('Matrícula: N° Cuotas', ''),
            original.get('Aranceles: N° Cuotas', ''),
            original.get('Matrícula: Monto por Cuota', ''),
            original.get('Aranceles: Monto Cuota Mínimo', ''),
            original.get('Aranceles: Monto Cuota Máximo', ''),
            original.get('Matrícula: Total Anual', ''),
            original.get('Aranceles: Total', ''),
        ])

print('rows_with_differences =', len(csv_internal_diffs))
print('issue_counts =', dict(csv_internal_counter))
print(csv_diff_json)
print(csv_diff_csv)
print(csv_diff_records_csv)

rows_with_differences = 11
issue_counts = {'CUOTAS_DISTINTAS': 8, 'MONTO_DISTINTO': 7, 'TOTAL_DISTINTO': 5, 'FEE_FALTANTE': 3}
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.json
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff.csv
c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv


In [9]:
import csv
from pathlib import Path

source_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy.csv')
diff_records_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv')
output_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy_11_casos.csv')

with source_csv.open('r', encoding='utf-8-sig', newline='') as f:
    source_rows = list(csv.DictReader(f))
    headers = list(source_rows[0].keys())

with diff_records_csv.open('r', encoding='utf-8-sig', newline='') as f:
    diff_rows = list(csv.DictReader(f))

target_ids = {row['ID Estudiante'] for row in diff_rows}
subset_rows = [row for row in source_rows if row['ID Estudiante'] in target_ids]

with output_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(subset_rows)

print('selected_rows =', len(subset_rows))
print(output_csv)
for row in subset_rows:
    print(row['ID Estudiante'], '|', row['Estudiante'])

selected_rows = 11
c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy_11_casos.csv
77e94297-b89a-4d0d-829e-8aa373449d14 | MIGUEL ESTEBAN VICUÑA PÉREZ
6c1e2c95-9d30-4eba-a4bc-a3dd5454bb17 | AMANDA FRANCISCA SALINAS VELIZ
5945ab89-9c94-4cd6-95ab-5a5767d2bb47 | MATIAS ANKATU CASTRO CORREA
66bf4d83-cb55-4acc-8658-c714f82c5f0e | VIOLETA BEATRIZ GUERRERO BIRKE
195577c6-a45c-4e97-9de3-76ebc4af4f09 | AMANDA AYELEN PAREDES RIOS
5a907b42-8a3a-4742-9aa1-4b02b6884045 | RAMIRO PASCUAL FUENZALIDA ANDRADE
3317705f-7dd3-49ca-b547-21b3155fd611 | RENATA EMILIA IRARRAZABAL BASAEZ
4c57b31c-4a96-4ac5-88ff-cdb827f4d862 | DAMIAN IGNACIO MIRANDA ALVAREZ
b4719a36-da47-4bbd-9950-8755d3c195b7 | FLORENCIA GALAZ DURÁN
97e49951-e13c-4438-856b-5a9b6db38602 | BASTIAN ANGEL MAICKOLL LOYOLA GRONDONA
bf024ee2-4e96-4c70-97c3-584d5669ddb1 | LUCAS MIGUEL CARREÑO SOTO


In [ ]:
import csv
from decimal import Decimal, InvalidOperation
from pathlib import Path

source_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy_11_casos.csv')
diff_records_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\tmp_csv_internal_matricula_vs_aranceles_diff_records.csv')
output_csv = Path(r'c:\Meik_Apps\winterhill-cobros-1.1\reporte_matriculados_forma_pago_vs_aranceles_2026_20260408 copy_11_casos_con_esperados.csv')

with source_csv.open('r', encoding='utf-8-sig', newline='') as f:
    source_rows = list(csv.DictReader(f))
    source_headers = list(source_rows[0].keys())

with diff_records_csv.open('r', encoding='utf-8-sig', newline='') as f:
    diff_rows = list(csv.DictReader(f))
diff_by_student = {row['ID Estudiante']: row for row in diff_rows}

def to_decimal(value):
    text = '' if value is None else str(value).strip()
    if not text:
        return None
    try:
        return Decimal(text)
    except (InvalidOperation, ValueError):
        return None

def is_positive(value):
    dec = to_decimal(value)
    return dec is not None and dec > 0

extra_headers = [
    'issues',
    'resolucion_sugerida',
    'esperado_matricula_forma_pago',
    'esperado_matricula_cuotas',
    'esperado_matricula_monto_por_cuota',
    'esperado_matricula_total_anual',
    'esperado_aranceles_n_cuotas',
    'esperado_aranceles_forma_pago',
    'esperado_aranceles_monto_min',
    'esperado_aranceles_monto_max',
    'esperado_aranceles_total',
]

output_rows = []
for row in source_rows:
    diff = diff_by_student[row['ID Estudiante']]
    issues = diff['issues'].split('|') if diff['issues'] else []
    arancel_has_values = (
        row.get('Aranceles: Forma de Pago', '').strip() not in ('', 'SIN_FEE')
        or is_positive(row.get('Aranceles: N° Cuotas', ''))
        or is_positive(row.get('Aranceles: Monto Cuota Mínimo', ''))
        or is_positive(row.get('Aranceles: Total', ''))
    )

    if 'FEE_FALTANTE' in issues and not arancel_has_values:
        resolucion = 'Copiar matrícula hacia aranceles'
    elif 'FEE_FALTANTE' in issues and arancel_has_values:
        resolucion = 'Revisión manual'
    else:
        resolucion = 'Copiar aranceles hacia matrícula'

    expected = dict(row)
    if resolucion == 'Copiar matrícula hacia aranceles':
        expected_aranceles_n_cuotas = row.get('Matrícula: N° Cuotas', '')
        expected_aranceles_forma_pago = row.get('Matrícula: Forma de Pago', '')
        expected_aranceles_monto = row.get('Matrícula: Monto por Cuota', '')
        expected_aranceles_total = row.get('Matrícula: Total Anual', '')
        expected_matricula_forma_pago = row.get('Matrícula: Forma de Pago', '')
        expected_matricula_cuotas = row.get('Matrícula: N° Cuotas', '')
        expected_matricula_monto = row.get('Matrícula: Monto por Cuota', '')
        expected_matricula_total = row.get('Matrícula: Total Anual', '')
    else:
        expected_matricula_forma_pago = row.get('Aranceles: Forma de Pago', '') if row.get('Aranceles: Forma de Pago', '').strip() else row.get('Matrícula: Forma de Pago', '')
        expected_matricula_cuotas = row.get('Aranceles: N° Cuotas', '') if row.get('Aranceles: N° Cuotas', '').strip() else row.get('Matrícula: N° Cuotas', '')
        expected_matricula_monto = row.get('Aranceles: Monto Cuota Mínimo', '') if row.get('Aranceles: Monto Cuota Mínimo', '').strip() else row.get('Matrícula: Monto por Cuota', '')
        expected_matricula_total = row.get('Aranceles: Total', '') if row.get('Aranceles: Total', '').strip() else row.get('Matrícula: Total Anual', '')
        expected_aranceles_n_cuotas = row.get('Aranceles: N° Cuotas', '')
        expected_aranceles_forma_pago = row.get('Aranceles: Forma de Pago', '')
        expected_aranceles_monto = row.get('Aranceles: Monto Cuota Mínimo', '')
        expected_aranceles_total = row.get('Aranceles: Total', '')

    output_row = dict(row)
    output_row.update({
        'issues': diff['issues'],
        'resolucion_sugerida': resolucion,
        'esperado_matricula_forma_pago': expected_matricula_forma_pago,
        'esperado_matricula_cuotas': expected_matricula_cuotas,
        'esperado_matricula_monto_por_cuota': expected_matricula_monto,
        'esperado_matricula_total_anual': expected_matricula_total,
        'esperado_aranceles_n_cuotas': expected_aranceles_n_cuotas,
        'esperado_aranceles_forma_pago': expected_aranceles_forma_pago,
        'esperado_aranceles_monto_min': expected_aranceles_monto,
        'esperado_aranceles_monto_max': expected_aranceles_monto,
        'esperado_aranceles_total': expected_aranceles_total,
    })
    output_rows.append(output_row)

with output_csv.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=source_headers + extra_headers, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(output_rows)

print('selected_rows =', len(output_rows))
print(output_csv)